# AMPS II · Week 14 — Lab
## Phase separation and parallel performance: energy-stable Cahn–Hilliard, exact growth rates, solvers, ghosts and a reused factorisation

| | |
|---|---|
| **Lecture sections** | Part A (Cahn–Hilliard extended: schemes, exact checks) · Part B (Flory–Huggins) · Part C (performance: solvers, communication, reuse) · Part D (LLM) |
| **Time** | ≈ 80 min in class |
| **You will** | compute growth rates and amplification factors in NumPy · measure each scheme's growth rate against its own discrete prediction · push three schemes to large time steps · time four linear solvers and look at the ghosts · reuse one factorisation for a whole run · judge an LLM's Cahn–Hilliard script |

**Reminder.** `%%fenicsx` cells are separate MPI programs; ordinary cells have NumPy and matplotlib only. The model module `phase.py` is written once with `%%writefile` (§1). All cells run with `-np 2`.
Timings depend on the machine — compare ratios, not seconds.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · The model module  *(lecture §A–C)*
`phase.py`: `box`, `noise(x)` (a smooth random field that is the same on every partition), `CahnHilliard(msh, eps, dt, scheme="be" | "cs" | "stab", S, potential="quartic" | "fh", chi)` with `.set_c`, `.step`, `.energy`, `.mass`,
`.mode(k)`, `.c()`; `CahnHilliardLinear` (the stabilised scheme with one factorisation); references `growth_rate`, `tanh_profile`, `fh_binodal`; performance probes `solve_poisson(N, method)`, `ghost_stats`, `wtime`. Run the cell.

In [ ]:
%%writefile /tmp/fnx/phase.py
"""phase.py — Cahn-Hilliard phase separation with energy-stable time stepping, and parallel-performance probes, DOLFINx 0.11 (AMPS II, Week 14)."""
import gc, time
import numpy as np
import ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem, mesh, geometry
from dolfinx.fem.petsc import NonlinearProblem
import dolfinx.fem.petsc as fp
import basix.ufl

comm = MPI.COMM_WORLD
gc.disable()                                   # PETSc destructors are collective: collect at the same point on all ranks
def collect():
    gc.collect()
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)
LU = {"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}
NEWTON = dict(LU, snes_type="newtonls", snes_linesearch_type="none", snes_rtol=1e-10, snes_atol=1e-12, snes_stol=0.0, snes_max_it=25)

def box(L=1.0, H=1.0, nx=32, ny=32, x0=0.0):
    return mesh.create_rectangle(comm, [[x0, 0], [x0 + L, H]], [nx, ny], mesh.CellType.quadrilateral)

def noise(x, amp=0.05, seed=1, modes=40):
    """a smooth random field as a function of the coordinates (the same on every partition, unlike np.random per dof)"""
    r = np.random.default_rng(seed); s = np.zeros(x.shape[1])
    for _ in range(modes):
        kx, ky = r.integers(1, 12, 2); ph = r.uniform(0, 2 * np.pi, 2)
        s += np.cos(kx * np.pi * x[0] + ph[0]) * np.cos(ky * np.pi * x[1] + ph[1])
    return amp * s / 10

def line(msh, f, y=0.5, x0=0.0, x1=1.0, n=401):
    """x and f(x, y) along a horizontal line (every rank gets all values)"""
    P = np.zeros((n, 3)); P[:, 0] = np.linspace(x0, x1, n); P[:, 1] = y
    tdim = msh.topology.dim; nloc = msh.topology.index_map(tdim).size_local
    cand = geometry.compute_colliding_cells(msh, geometry.compute_collisions_points(geometry.bb_tree(msh, tdim), P), P)
    out, cnt = np.zeros(n), np.zeros(n); idx, cells = [], []
    for j in range(n):
        own = [c for c in cand.links(j) if c < nloc]
        if own:
            idx.append(j); cells.append(own[0])
    if idx:
        out[idx] = np.asarray(f.eval(P[idx], np.array(cells, dtype=np.int32))).reshape(len(idx), -1)[:, 0]; cnt[idx] = 1
    return P[:, 0], comm.allreduce(out, op=MPI.SUM) / np.maximum(comm.allreduce(cnt, op=MPI.SUM), 1)

# ═══════════════════════════════════════════════ free energies
def quartic(c):
    """f(c) = (c^2 - 1)^2 / 4 (phases c = -1, +1)"""
    return 0.25 * (c ** 2 - 1) ** 2

def flory_huggins(phi, chi, N1=1.0, N2=1.0):
    """f(phi) = phi/N1 ln phi + (1 - phi)/N2 ln(1 - phi) + chi phi (1 - phi) (lattice units)"""
    return phi / N1 * ufl.ln(phi) + (1 - phi) / N2 * ufl.ln(1 - phi) + chi * phi * (1 - phi)

class CahnHilliard:
    """c_t = div(M grad mu), mu = f'(c) - eps^2 lap c on a box with no-flux walls; mixed P1 x P1 (c, mu).
    scheme: "be" (fully implicit backward Euler, Newton), "cs" (convex splitting, Eyre: implicit convex part, explicit concave part),
            "stab" (linear, stabilised: f'(c_n) + S (c - c_n); one constant matrix), "cn" (mu averaged over the step: not energy stable).
    potential: "quartic" (f = (c^2-1)^2/4) or "fh" (Flory-Huggins with chi; c is the volume fraction phi in (0, 1))."""
    count = 0
    def __init__(self, msh, eps=0.05, M=1.0, dt=1e-4, scheme="cs", S=2.0, potential="quartic", chi=3.0, deg=1, solver=None):
        collect(); CahnHilliard.count += 1; k = CahnHilliard.count
        self.msh, self.scheme, self.potential, self.chi = msh, scheme, potential, chi
        cell = msh.basix_cell(); P = basix.ufl.element("Lagrange", cell, deg)
        W = self.W = fem.functionspace(msh, basix.ufl.mixed_element([P, P]))
        self.w, self.w0 = fem.Function(W), fem.Function(W)
        c, mu = ufl.split(self.w); c0, mu0 = ufl.split(self.w0); q, v = ufl.TestFunctions(W)
        self.dt = fem.Constant(msh, float(dt)); self.eps = eps; self.M = M
        if potential == "quartic":
            fc, fe = (lambda c: c ** 3), (lambda c: c)                  # f' = c^3 - c: convex part c^3, concave part -c
            fp_ = lambda c: c ** 3 - c
        else:                                                           # Flory-Huggins: convex part ln terms, concave part chi (1 - 2 phi)
            fc = lambda c: ufl.ln(c / (1 - c)) + 0 * c
            fe = lambda c: -chi * (1 - 2 * c)
            fp_ = lambda c: ufl.ln(c / (1 - c)) + chi * (1 - 2 * c)
        mob = mu                                                        # the chemical potential in the mobility term
        if scheme == "be":
            dfdc = fp_(c)
        elif scheme == "cn":                                            # theta = 1/2 on mu (as often written; NOT energy stable for Cahn-Hilliard)
            dfdc = fp_(c); mob = 0.5 * (mu + mu0)
        elif scheme == "cs":
            dfdc = fc(c) - fe(c0)
        elif scheme == "stab":
            dfdc = fp_(c0) + S * (c - c0)
        else:
            raise ValueError(scheme)
        F = ((c - c0) / self.dt * q + M * ufl.dot(ufl.grad(mob), ufl.grad(q)) + (mu - dfdc) * v - eps ** 2 * ufl.dot(ufl.grad(c), ufl.grad(v))) * ufl.dx
        self.F = F
        opts = dict(NEWTON) if solver is None else dict(NEWTON, **solver)
        self.pb = NonlinearProblem(F, self.w, petsc_options_prefix=f"ch{k}_", petsc_options=opts)
        self.Vc, self.mapc = W.sub(0).collapse()
        self.t = 0.0; self.its = []

    def set_c(self, fun):
        """initial composition c(x) (numpy callable); mu is computed from it at the first solve"""
        cf = fem.Function(self.Vc); cf.interpolate(fun)
        self.w.x.array[self.mapc] = cf.x.array; self.w.x.scatter_forward()
        self.w0.x.array[:] = self.w.x.array

    def step(self):
        self.w0.x.array[:] = self.w.x.array
        self.pb.solve(); s = self.pb.solver; r = int(s.getConvergedReason()); n = int(s.getIterationNumber())
        self.t += float(self.dt.value); self.its.append(n)
        return r, n

    def c(self):
        return self.w.sub(0).collapse()

    def energy(self):
        c = ufl.split(self.w)[0]
        f = quartic(c) if self.potential == "quartic" else flory_huggins(c, self.chi)
        return comm.allreduce(fem.assemble_scalar(fem.form((f + 0.5 * self.eps ** 2 * ufl.dot(ufl.grad(c), ufl.grad(c))) * ufl.dx)), op=MPI.SUM)

    def mass(self):
        c = ufl.split(self.w)[0]
        return comm.allreduce(fem.assemble_scalar(fem.form(c * ufl.dx)), op=MPI.SUM)

    def mode(self, k, L=1.0):
        """amplitude of cos(k x) in c: 2/(L H) int c cos(k x)"""
        c = ufl.split(self.w)[0]; x = ufl.SpatialCoordinate(self.msh)
        area = comm.allreduce(fem.assemble_scalar(fem.form(1.0 * ufl.dx(domain=self.msh))), op=MPI.SUM)
        return 2 / area * comm.allreduce(fem.assemble_scalar(fem.form(c * ufl.cos(k * x[0]) * ufl.dx)), op=MPI.SUM)

def growth_rate(k, eps, M=1.0, c0=0.0, fpp=None):
    """linear growth rate of a mode cos(k x) about c0: sigma = -M k^2 (f''(c0) + eps^2 k^2); f'' = 3 c0^2 - 1 (quartic)"""
    fpp = 3 * c0 ** 2 - 1 if fpp is None else fpp
    return -M * k ** 2 * (fpp + eps ** 2 * k ** 2)

def tanh_profile(x, eps):
    """1D equilibrium interface of the quartic potential: c = tanh(x / (sqrt(2) eps)); interfacial energy sigma = 2 sqrt(2) eps / 3"""
    return np.tanh(x / (np.sqrt(2) * eps))

def fh_binodal(chi, tol=1e-14):
    """symmetric Flory-Huggins (N1 = N2 = 1): coexisting phi from ln(phi/(1 - phi)) + chi (1 - 2 phi) = 0, phi < 1/2 (bisection)"""
    if chi <= 2:
        return 0.5, 0.5
    g = lambda p: np.log(p / (1 - p)) + chi * (1 - 2 * p)
    a, b = 1e-300, 0.5 - 1e-12
    for _ in range(200):
        m = 0.5 * (a + b)
        if g(a) * g(m) <= 0:
            b = m
        else:
            a = m
        if b - a < tol:
            break
    p = 0.5 * (a + b)
    return p, 1 - p

# ═══════════════════════════════════════════════ parallel performance
def wtime():
    comm.Barrier(); return MPI.Wtime()

def poisson(N, dim=3, deg=1):
    """-lap u = f on the unit square/cube, u = 0 on the boundary, u = exp(x + y) prod x_i (1 - x_i): returns (V, a, L, bcs, u_exact)"""
    msh = (mesh.create_unit_cube(comm, N, N, N, mesh.CellType.hexahedron) if dim == 3
           else mesh.create_unit_square(comm, N, N, mesh.CellType.quadrilateral))
    V = fem.functionspace(msh, ("Lagrange", deg)); u, v = ufl.TrialFunction(V), ufl.TestFunction(V); x = ufl.SpatialCoordinate(msh)
    ue = ufl.exp(x[0] + x[1])                                       # not an eigenfunction of the discrete operator (a sine product is,
    for i in range(dim):                                            # on a uniform tensor mesh: CG would then converge in one step)
        ue = ue * x[i] * (1 - x[i])
    f = -ufl.div(ufl.grad(ue))
    fdim = dim - 1; bnd = mesh.locate_entities_boundary(msh, fdim, lambda x: np.full(x.shape[1], True))
    bc = fem.dirichletbc(0.0, fem.locate_dofs_topological(V, fdim, bnd), V)
    return V, ufl.dot(ufl.grad(u), ufl.grad(v)) * ufl.dx, f * v * ufl.dx, [bc], ue

def solve_poisson(N, method="gamg", dim=3, deg=1, rtol=1e-8):
    """times (max over ranks) of assembly, preconditioner setup and solve, iterations and the L2 error.
    method: "lu" (MUMPS), "gamg" (CG + algebraic multigrid), "jacobi" (CG + Jacobi), "mf" (matrix-free CG, no matrix, no preconditioner)"""
    collect()
    V, a, L, bcs, ue = poisson(N, dim, deg); msh = V.mesh; uh = fem.Function(V)
    out = dict(N=N, dofs=V.dofmap.index_map.size_global, ranks=comm.size, method=method)
    t0 = wtime(); b = fp.assemble_vector(fem.form(L)); fp.apply_lifting(b, [fem.form(a)], [bcs]); b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    fp.set_bc(b, bcs)
    ksp = PETSc.KSP().create(comm)
    if method == "mf":
        wf = fem.Function(V); actf = fem.form(ufl.action(a, wf)); yg = fp.create_vector(V)
        nown = V.dofmap.index_map.size_local * V.dofmap.index_map_bs
        bdofs = bcs[0].dof_indices()[0]; bown = bdofs[bdofs < nown]
        class Shell:
            """y = A x without a matrix: assemble the action of the bilinear form on x (into a ghosted work vector)"""
            def mult(self, mat, X, Y):
                wf.x.array[:nown] = X.array_r; wf.x.scatter_forward()
                with yg.localForm() as yl:
                    yl.set(0.0)
                fp.assemble_vector(yg, actf); yg.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
                y = Y.array; y[:] = yg.array_r[:nown]; y[bown] = X.array_r[bown]      # identity rows for the Dirichlet dofs
        A = PETSc.Mat().createPython([[nown, PETSc.DETERMINE], [nown, PETSc.DETERMINE]], Shell(), comm=comm); A.setUp()
        t1 = wtime(); out["assemble"] = t1 - t0; out["nnz"] = 0
        ksp.setOperators(A); ksp.setType("cg"); ksp.getPC().setType("none")
    else:
        A = fp.assemble_matrix(fem.form(a), bcs=bcs); A.assemble()
        t1 = wtime(); out["assemble"] = t1 - t0; out["nnz"] = A.getInfo(PETSc.Mat.InfoType.GLOBAL_SUM)["nz_used"]
        ksp.setOperators(A)
        if method == "lu":
            ksp.setType("preonly"); ksp.getPC().setType("lu"); ksp.getPC().setFactorSolverType("mumps")
        else:
            ksp.setType("cg"); ksp.getPC().setType("gamg" if method == "gamg" else "jacobi")
    ksp.setTolerances(rtol=rtol, atol=1e-50, max_it=20000)
    t2 = wtime(); ksp.setUp(); t3 = wtime(); out["setup"] = t3 - t2
    ksp.solve(b, uh.x.petsc_vec); uh.x.scatter_forward(); t4 = wtime(); out["solve"] = t4 - t3
    out["its"] = ksp.getIterationNumber(); out["reason"] = ksp.getConvergedReason()
    out["error"] = np.sqrt(comm.allreduce(fem.assemble_scalar(fem.form((uh - ue) ** 2 * ufl.dx)), op=MPI.SUM))
    ksp.destroy(); A.destroy(); b.destroy()
    if method == "mf":
        yg.destroy()
    return out

def ghost_stats(msh):
    """per-rank owned and ghost cells and vertices, and the number of neighbouring ranks (gathered on every rank)"""
    tdim = msh.topology.dim; msh.topology.create_connectivity(0, tdim)
    ic, iv = msh.topology.index_map(tdim), msh.topology.index_map(0)
    loc = [ic.size_local, ic.num_ghosts, iv.size_local, iv.num_ghosts, len(set(iv.owners.tolist()))]
    return np.array(comm.allgather(loc))

class CahnHilliardLinear:
    """the stabilised scheme as a LINEAR problem: the matrix (c, mu) is constant, assembled and factorised ONCE; each step
    assembles the right-hand side and does two triangular solves. Same result as CahnHilliard(scheme="stab")."""
    count = 0
    def __init__(self, msh, eps=0.05, M=1.0, dt=1e-4, S=2.0, deg=1, reuse=True):
        collect(); CahnHilliardLinear.count += 1; k = CahnHilliardLinear.count
        self.msh, self.eps, self.reuse = msh, eps, reuse
        cell = msh.basix_cell(); P = basix.ufl.element("Lagrange", cell, deg)
        W = self.W = fem.functionspace(msh, basix.ufl.mixed_element([P, P]))
        self.w, self.w0 = fem.Function(W), fem.Function(W)
        (c, mu), (q, v) = ufl.TrialFunctions(W), ufl.TestFunctions(W); c0 = ufl.split(self.w0)[0]
        a = (c / dt * q + M * ufl.dot(ufl.grad(mu), ufl.grad(q)) + mu * v - S * c * v - eps ** 2 * ufl.dot(ufl.grad(c), ufl.grad(v))) * ufl.dx
        L = (c0 / dt * q + (c0 ** 3 - c0 - S * c0) * v) * ufl.dx
        self.a, self.L = fem.form(a), fem.form(L)
        self.A = fp.assemble_matrix(self.a); self.A.assemble(); self.b = fp.create_vector(W)
        self.ksp = PETSc.KSP().create(comm); self.ksp.setOperators(self.A); self.ksp.setType("preonly")
        self.ksp.getPC().setType("lu"); self.ksp.getPC().setFactorSolverType("mumps"); self.ksp.setOptionsPrefix(f"chl{k}_")
        self.Vc, self.mapc = W.sub(0).collapse(); self.t = 0.0; self.dt = dt

    def set_c(self, fun):
        cf = fem.Function(self.Vc); cf.interpolate(fun); self.w.x.array[self.mapc] = cf.x.array; self.w.x.scatter_forward()

    def step(self):
        self.w0.x.array[:] = self.w.x.array
        if not self.reuse:                                              # re-assemble and re-factorise every step (what a Newton loop does)
            self.A.zeroEntries(); fp.assemble_matrix(self.A, self.a); self.A.assemble(); self.ksp.setOperators(self.A)
        with self.b.localForm() as bl:
            bl.set(0.0)
        fp.assemble_vector(self.b, self.L); self.b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
        self.ksp.solve(self.b, self.w.x.petsc_vec); self.w.x.scatter_forward(); self.t += self.dt

    def energy(self):
        c = ufl.split(self.w)[0]
        return comm.allreduce(fem.assemble_scalar(fem.form((quartic(c) + 0.5 * self.eps ** 2 * ufl.dot(ufl.grad(c), ufl.grad(c))) * ufl.dx)), op=MPI.SUM)

## 2 · Growth rates and amplification factors in NumPy  *(lecture §A2–A3)*
Linearised about $c_0=0$ ($f''(0)=-1$), a mode $\cos kx$ grows like $e^{\sigma t}$, $\sigma=Mk^2(1-\varepsilon^2k^2)$. One time step multiplies it by a factor $g$ that depends on the scheme.

In [ ]:
import numpy as np
eps, M = 0.05, 1.0
kmax = 1 / (np.sqrt(2) * eps); print(f"fastest mode k = {kmax:.3f} (wavelength {2 * np.pi / kmax:.4f}), sigma_max = {1 / (4 * eps ** 2):.2f}; modes with k > 1/eps = {1 / eps:.1f} decay")
k = 4 * np.pi; k2 = k * k; sig = M * k2 * (1 - eps ** 2 * k2)
for dt in (4e-4, 1e-4, 1e-2):
    g = {"exact": np.exp(sig * dt), "be": 1 / (1 - dt * sig), "cs": (1 + dt * k2) / (1 + dt * eps ** 2 * k2 ** 2),
         "stab (S=2)": (1 + dt * k2 * 3) / (1 + dt * k2 * (2 + eps ** 2 * k2))}
    print(f"dt {dt:.0e}: " + ", ".join(f"{n} g = {v:.4f} (sigma_h = {np.log(abs(v)) / dt:.2f})" for n, v in g.items()))

> **✏️ Your turn (4 min).** (i) At $\Delta t=10^{-2}$ backward Euler gives $g<0$. What does that mean for the mode? (ii) Which schemes keep $|g|$ bounded for every $\Delta t$, and does that make them accurate?

> *your answer*

## 3 · Each scheme against its own prediction  *(lecture §A3)*
A thin strip ($128\times2$ cells, $\varepsilon=0.05$), $c=10^{-4}\cos4\pi x$, $\Delta t=2\times10^{-4}$ up to $t=0.02$. The measured growth rate against the exact $\sigma$ and against the scheme's discrete rate from §2.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from phase import *

eps = 0.05; k = 4 * np.pi; k2 = k * k; sig = growth_rate(k, eps); dt = 2e-4
disc = {"be": -np.log(1 - dt * sig) / dt, "cs": np.log((1 + dt * k2) / (1 + dt * eps ** 2 * k2 * k2)) / dt, "stab": np.log((1 + dt * k2 * 3) / (1 + dt * k2 * (2 + eps ** 2 * k2))) / dt}
for scheme in ("be", "cs", "stab"):
    msh = box(1.0, 0.05, 128, 2); ch = CahnHilliard(msh, eps=eps, dt=dt, scheme=scheme, S=2.0)
    ch.set_c(lambda x: 1e-4 * np.cos(k * x[0])); a0 = ch.mode(k); t0 = wtime()
    for i in range(100):
        ch.step()
    s = np.log(ch.mode(k) / a0) / ch.t
    say(f"{scheme:4s}: sigma_h {s:.4f} | exact {sig:.4f} ({100 * (s / sig - 1):+.2f} %) | its own discrete rate {disc[scheme]:.4f} ({100 * (s / disc[scheme] - 1):+.3f} %) | Newton its/step {np.mean(ch.its):.1f}, {wtime() - t0:.1f} s")

> **✏️ Your turn (4 min).** What does the agreement with the discrete rate verify, and what does the deviation from the exact $\sigma$ measure? Why is the stabilised scheme the least accurate?

> *your answer*

## 4 · Large time steps: stable, failing, or frozen  *(lecture §A4)*
$64\times64$, $\varepsilon=0.02$, the same smooth random start, up to $t=0.05$ with $\Delta t=10^{-3}$ and $10^{-2}$. Reference (backward Euler, $\Delta t=10^{-4}$, lecture): $E(0.05)=0.12970$.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from phase import *

for scheme in ("be", "cs", "stab"):
    for dt in (1e-3, 1e-2):
        msh = box(1, 1, 64, 64); ch = CahnHilliard(msh, eps=0.02, dt=dt, scheme=scheme, S=2.0); ch.set_c(noise)
        E = [ch.energy()]; fails = 0
        for i in range(int(round(0.05 / dt))):
            r = ch.step(); fails += r[0] <= 0; E.append(ch.energy())
        inc = sum(1 for a, b in zip(E, E[1:]) if b > a + 1e-12)
        say(f"{scheme:4s} dt {dt:.0e}: E(0.05) {E[-1]:.5f} (reference 0.12970), max E {max(E):.4g}, energy increases {inc}, Newton failures {fails}, max its {max(ch.its)}")

> **✏️ Your turn (5 min).** Rank the three schemes at $\Delta t=10^{-2}$: which one would you trust, which one fails loudly, which one fails quietly?

> *your answer*

## 5 · Solvers, ghosts and a reused factorisation  *(lecture §C)*
(a) A 3D Poisson problem (Q1 hexahedra) with four solvers. (b) The ghosts of a 2-rank partition. (c) The stabilised Cahn–Hilliard scheme with Newton (re-assembles and re-factorises every step) against one factorisation reused.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from phase import *

solve_poisson(8, "gamg")                                                          # JIT warm-up (the first call compiles the forms)
for N in (16, 24, 32):
    out = []
    for m in ("lu", "gamg", "jacobi", "mf"):
        o = solve_poisson(N, m); out.append(f"{m} {o['setup'] + o['solve']:.3f} s ({o['its']} its)")
        nnz = o["nnz"] if m == "lu" else nnz
    say(f"N {N} ({o['dofs']} unknowns, {nnz:.0f} matrix nonzeros): " + " | ".join(out))
V, *_ = poisson(32); g = ghost_stats(V.mesh)
say("per rank [owned cells, ghost cells, owned vertices, ghost vertices, neighbours]:", g.tolist())
for name in ("newton-stab", "linear-refactor", "linear-reuse"):
    msh = box(1, 1, 64, 64)
    ch = CahnHilliard(msh, eps=0.02, dt=1e-4, scheme="stab") if name == "newton-stab" else CahnHilliardLinear(msh, eps=0.02, dt=1e-4, reuse=(name == "linear-reuse"))
    ch.set_c(noise); ch.step(); t0 = wtime()
    for i in range(20):
        ch.step()
    say(f"{name:16s}: {1e3 * (wtime() - t0) / 20:6.1f} ms per step, E {ch.energy():.8f}")

> **✏️ Your turn (6 min).** (i) Which solver's iterations grow with $N$, and how? (ii) What fraction of the vertices of each rank are ghosts, and what does that cost in each CG iteration? (iii) Why can the stabilised scheme reuse one factorisation, and backward Euler not?

> *your answer*

## 6 · LLM exercise: spinodal decomposition  *(lecture §D2)*
Ask an LLM for a DOLFINx 0.11 script that simulates spinodal decomposition with the Cahn–Hilliard equation ($\varepsilon=0.02$, $M=1$, mean $c=0$, unit square, no-flux walls) with an **energy-stable** time stepping, and
that verifies itself. Paste it below, run it with `-np 2` (and `-np 1`), and judge it against §3–§4.

In [ ]:
%%fenicsx -np 2
# Paste the LLM's script here (keep the %%fenicsx line).

> *your verdict on the LLM's code*